# Longitudinal flow results: geometry, volume, velocity, and disease effects

Primary panels compare completed current cocycle models on the identical test-set intersection. Existing completed ODE and BrainODE experiments appear later as a separate reference analysis. Smoke runs are excluded.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
RELATIVE_TASK = Path('examples/ADNI_1_LHipp_LLV_No_MCI_synthseg_minimal_smooth/task4_all_flow_visual_analysis_v1')
TASK = next((root / RELATIVE_TASK for root in (Path.cwd(), *Path.cwd().parents) if (root / RELATIVE_TASK).exists()), None)
if TASK is None: raise FileNotFoundError('Could not locate the task directory from the notebook working directory')
sys.path.insert(0, str(TASK / 'scripts'))
import notebook_helpers as H
CACHE = Path('/mnt/bulk10tb/Deep3DComp/ADNI_1_LHipp/task4_all_flow_visual_analysis_v1/cache_v1')
D = H.load_cache(CACHE)
MANIFEST = json.loads((CACHE / 'manifest.json').read_text())
display(Markdown(f"**Loaded:** {MANIFEST['current_endpoint_rows']} endpoint rows, {MANIFEST['current_velocity_rows']} velocity rows, and {MANIFEST['shared_subjects']} shared test subjects."))

## 1. Cohort and checkpoint audit

In [ ]:
audit = D['endpoint'].groupby(['method_label','diagnosis'], as_index=False).agg(subjects=('subject_id','nunique'), pairs=('subject_id','size'), mean_followup_years=('followup_years','mean'))
display(audit)
display(D['inventory'][['method_label','checkpoint_epoch','status','primary']])

## 2. Endpoint surface accuracy and overlap

In [ ]:
H.metric_bars(D, 'assd_mm', title='Sampled average symmetric surface distance').show()
H.metric_bars(D, 'dice', title='Volumetric Dice').show()

In [ ]:
H.metric_bars(D, 'hd95_mm', title='Robust Hausdorff distance').show()
H.metric_bars(D, 'chamfer_l2_squared_mm2', title='Chamfer distance').show()

## 3. Improvement over no change and representation floor

In [ ]:
H.improvement_forest(D, ['assd_mm','hd95_mm','volume_relative_error','rate_absolute_error_per_year','dice','hotspot_dice']).show()

## 4. Local anatomical-change accuracy

In [ ]:
H.metric_bars(D, 'normal_change_mae_mm_per_year', title='Local normal-change error').show()
H.metric_bars(D, 'hotspot_dice', title='Top-change hotspot Dice').show()
H.metric_bars(D, 'normal_change_pearson', title='Local-change correlation').show()

## Cocycle consistency

In [ ]:
H.consistency_plot(D).show()
display(D['consistency'])

## 5. CN and AD volume trends

In [ ]:
H.volume_trends(D).show()
H.volume_calibration(D).show()

In [ ]:
trend = D['endpoint'].groupby(['method_label','diagnosis'], as_index=False).agg(predicted=('predicted_signed_log_volume_rate_per_year','mean'), observed=('observed_signed_log_volume_rate_per_year','mean'), rate_mae=('rate_absolute_error_per_year','mean'), direction=('atrophy_direction_agreement','mean'))
trend['predicted_percent_per_year'] = 100*np.expm1(trend.predicted)
trend['observed_percent_per_year'] = 100*np.expm1(trend.observed)
display(trend[['method_label','diagnosis','predicted_percent_per_year','observed_percent_per_year','rate_mae','direction']])

## 6. AD-minus-CN disease-gap recovery

In [ ]:
pivot = trend.pivot(index='method_label', columns='diagnosis', values=['predicted','observed'])
gap = pd.DataFrame({'predicted_gap': pivot['predicted']['AD']-pivot['predicted']['CN'], 'observed_gap': pivot['observed']['AD']-pivot['observed']['CN']})
gap['gap_recovery_percent'] = 100*gap.predicted_gap/gap.observed_gap
gap['absolute_gap_error'] = (gap.predicted_gap-gap.observed_gap).abs()
display(gap.sort_values('absolute_gap_error'))

## 7. Instantaneous velocity: definitions and limits

For an encoded shape $z$ at age $t$ and condition $d$, the model-implied latent generator is

$$v_\theta(z,t,d)=\left.\partial_\tau\Phi_\theta(z,t,\tau,d)\right|_{\tau=t}.$$

For DirectC4, $\Phi(z,s,t,d)=z+(t-s)\phi(z,s,t,d)$, so the diagonal value is $v_\theta(z,t,d)=\phi(z,t,t,d)$. It describes what the fitted model predicts at that state; it is not a directly measured biological velocity.

The plots use **Observed (GT estimate)** for the dataset reference. At an interior visit it is computed from the recorded scans immediately before and after that visit; endpoints use the nearest recorded interval. Therefore, “GT” in the plot labels means *computed from observed longitudinal scans*. It is not a direct measurement of continuous instantaneous motion between scans. Current intervals range from about 0.47 to 3.36 years.

Latent results use standardized-coordinate units per year and should be interpreted within each representation. Surface results decode a 0.05-year model step and report signed normal motion in mm/year. For INR, the equivalent implicit level-set derivative is used. Negative surface-normal velocity means inward motion; positive means outward motion.


### Latent speed magnitude

**RMS standardized coordinate/year** is the square root of the mean squared velocity across latent coordinates. It measures total latent-change speed without displaying latent width. The horizontal axis is the Observed (GT estimate); the vertical axis is the model generator. The dashed identity line indicates equal magnitude. Points below it mean the model changes more slowly than the observed estimate.


In [ ]:
H.velocity_scatter(D).show()


### Latent direction agreement

**Cosine similarity** compares the directions of the full latent velocity vectors and ignores their magnitudes. A value of $+1$ means the model and Observed (GT estimate) point in the same latent direction, $0$ means no directional alignment, and $-1$ means opposite directions.


In [ ]:
H.velocity_alignment(D).show()


### Latent speed trend with age

Each point is the median latent RMS speed within an age band. Solid/dashed series distinguish the model generator from the Observed (GT estimate), while the two panels separate CN and AD. This shows whether speed rises or falls with age; it does not establish a continuously observed trajectory between visits.


In [ ]:
H.velocity_by_age(D).show()


### Same-shape latent condition effect

For every observed shape and age, the model is evaluated twice: once with the CN condition and once with the AD condition. The metric is the RMS magnitude of $v_{AD}-v_{CN}$ per coordinate/year. Zero means the condition input does not change the local field. This is a controlled model sensitivity, not the difference between the actual AD and CN subject groups.


In [ ]:
H.condition_velocity_gap(D).show()


### Surface speed trend with age

**Area-weighted RMS normal speed (mm/year)** measures the magnitude of inward/outward motion across the whole surface. Squaring removes direction, so both contraction and expansion increase this value. Model and Observed (GT estimate) use the same physical unit and can be compared across representations.


In [ ]:
H.surface_speed_by_age(D).show()


### Signed surface trend with age

**Area-weighted mean normal velocity (mm/year)** preserves direction. Negative values indicate average inward movement or contraction; positive values indicate average outward movement or expansion. The zero line marks no net signed surface movement. Local inward and outward changes can cancel in this mean, so it should be read together with RMS speed.


In [ ]:
H.surface_signed_trend_by_age(D).show()


### Surface agreement metrics

Four complementary metrics are shown:

- **MAE (mm/year):** mean absolute vertex-wise normal-velocity error; lower is better.
- **Spatial Pearson correlation:** similarity of the surface pattern after removing overall offset and scale; $+1$ is best, $0$ indicates no linear pattern match, and $-1$ is reversed.
- **Sign agreement:** surface-area fraction where model and Observed (GT estimate) agree on inward versus outward motion; higher is better.
- **Speed ratio:** model RMS speed divided by Observed RMS speed; 1 is magnitude matched, below 1 is slower, and above 1 is faster.


In [ ]:
H.surface_velocity_agreement(D).show()


### Same-shape surface condition effect with age

This is the area-weighted RMS of the surface field obtained by subtracting CN-condition velocity from AD-condition velocity while holding shape and age fixed. Larger values mean stronger use of the disease-condition input. The CN and AD panels indicate whether that controlled sensitivity depends on the type of shape supplied to the model.


In [ ]:
H.surface_condition_effect_by_age(D).show()


### Spatial AD-minus-CN velocity pattern

Each mesh shows one average field per method. For both Observed (GT estimate) and models, the displayed value is the mean instantaneous normal velocity in AD subjects minus the mean in CN subjects at corresponding surface locations. Negative values mean AD is more inward-moving than CN; positive values mean AD is more outward-moving. This is an observed-group comparison and is different from switching the condition on the same shape.


In [ ]:
H.instantaneous_surface_group_maps(D).show()


### Internal diagonal-generator consistency check

This compares the analytic diagonal generator with the velocity obtained from a 0.05-year model transport step. The metric is RMS difference per coordinate/year on a logarithmic axis; smaller is better. It verifies numerical implementation and local continuity only. It does **not** measure agreement with observed anatomy.


In [ ]:
H.diagonal_check(D).show()
velocity_audit = D['velocity_summary'][[
    'method_label','diagnosis','velocity_rmse_per_coordinate_per_year_mean',
    'velocity_cosine_mean','diagonal_fd_rmse_per_coordinate_per_year_mean'
]].rename(columns={
    'method_label': 'Method', 'diagnosis': 'Diagnosis',
    'velocity_rmse_per_coordinate_per_year_mean': 'Model vs Observed RMS error/coordinate/year',
    'velocity_cosine_mean': 'Mean direction cosine',
    'diagonal_fd_rmse_per_coordinate_per_year_mean': 'Internal diagonal-step RMS difference/coordinate/year',
})
display(velocity_audit)


## 8. One averaged surface-change mesh per current method

In [ ]:
H.current_surface_maps(D).show()

## 9. Representation-family analyses

Ordinary plots intentionally hide latent width. This requested dedicated section separates the fixed-topology 128-dimensional family from the 256-dimensional implicit-field model.

In [ ]:
metrics = ['assd_mm','dice','volume_relative_error','rate_absolute_error_per_year','normal_change_mae_mm_per_year','hotspot_dice']
fixed = D['metric_summary'].query("diagnosis == 'overall' and method in ['pca','spiral','adaptive']")[['method_label']+metrics]
implicit = D['metric_summary'].query("diagnosis == 'overall' and method == 'inr'")[['method_label']+metrics]
display(Markdown('### Fixed-topology representation family'))
display(fixed)
display(Markdown('### Implicit-field representation family'))
display(implicit)

## 10. Existing completed Cocycle, ODE, and BrainODE results

This is an independent reference cohort and protocol. It is not pooled with, ranked against, or used for significance claims about the current matched analysis.

In [ ]:
H.legacy_endpoint_plot(D, 'assd').show()
H.legacy_horizon_plot(D, 'assd').show()


### Instantaneous velocity in the existing Cocycle/ODE/BrainODE cohort

These models use a different completed test cohort and protocol, so their values remain separate from the current matched analysis. Cocycle models use their diagonal generator; Latent ODE and BrainODE use their learned ODE vector field. **Observed (GT estimate)** has the same meaning as above: it is calculated from neighboring recorded scans, not directly measured continuous motion.


#### Latent speed magnitude

The axes show RMS standardized latent speed per coordinate/year. Dividing the standardized vector norm by the square root of coordinate count removes the direct width effect. The dashed line indicates equal model and Observed (GT estimate) magnitude. This normalization still does not make the different latent coordinate systems anatomically identical.


In [ ]:
H.legacy_velocity_plot(D).show()


#### Latent direction agreement

Cosine similarity measures whether the model vector and Observed (GT estimate) point in the same latent direction: $+1$ is the same direction, $0$ is no alignment, and $-1$ is opposite. It does not assess speed magnitude.


In [ ]:
H.legacy_velocity_alignment(D).show()


#### Latent speed trend with age

Each point is the median RMS standardized-coordinate speed in an age band. Model and Observed (GT estimate) lines are shown separately for CN and AD. Use this for within-architecture age trends, not for pooling this cohort with the current experiment.


In [ ]:
H.legacy_velocity_by_age(D).show()


#### Surface speed trend with age

Area-weighted RMS normal speed reports the magnitude of surface movement in mm/year and can be compared across PCA, INR, Latent ODE, and BrainODE decoders. It has no inward/outward sign.


In [ ]:
H.surface_speed_by_age(D, legacy=True).show()


#### Signed surface trend with age

Area-weighted mean normal velocity preserves direction: negative is inward contraction and positive is outward expansion. Because inward and outward regions can cancel, interpret this together with RMS surface speed.


In [ ]:
H.surface_signed_trend_by_age(D, legacy=True).show()


#### Surface agreement metrics

**MAE** is vertex-wise error in mm/year (lower is better); **Pearson correlation** measures spatial-pattern agreement; **sign agreement** is the area fraction with matching inward/outward direction; and **speed ratio** compares model RMS magnitude with Observed (GT estimate), with 1 indicating equal speed.


In [ ]:
H.surface_velocity_agreement(D, legacy=True).show()


#### Same-shape surface condition effect with age

The model is evaluated under AD and CN conditions at the same shape and age. The plotted RMS difference in mm/year measures controlled condition sensitivity. It is not the actual AD-subject minus CN-subject group difference.


In [ ]:
H.surface_condition_effect_by_age(D, legacy=True).show()


#### Spatial AD-minus-CN velocity pattern

Each mesh contains one average field. The value is mean AD-subject normal velocity minus mean CN-subject normal velocity at each corresponding location. Negative means more inward movement in AD; positive means more outward movement in AD. The Observed (GT estimate) mesh is calculated from the recorded scans.


In [ ]:
H.instantaneous_surface_group_maps(D, legacy=True).show()


## 11. One averaged surface-change mesh per existing architecture

In [ ]:
H.legacy_surface_maps(D).show()

## 12. Subject-bootstrap uncertainty

In [ ]:
key_metrics = ['assd_mm','dice','volume_relative_error','rate_absolute_error_per_year','hotspot_dice']
display(D['bootstrap'].query("diagnosis == 'overall' and metric in @key_metrics").sort_values(['metric','mean']))

### Paired subject-level tests with Holm correction

In [ ]:
display(D['paired_tests'].sort_values(['metric','holm_adjusted_p']))

## 13. Interpretation contract

- Primary comparisons are test-only and subject matched.
- Negative volume rate means atrophy.
- Latent velocity magnitude uses RMS per standardized coordinate; surface velocity uses mm/year.
- Observed (GT estimate) is calculated from recorded longitudinal scans and is not directly measured continuous motion.
- Surface maps average within subject before averaging across subjects.
- Existing ODE/BrainODE results remain a separate reference analysis.
- Surface distances are deterministic bidirectional sampled-surface nearest-neighbour estimates.
- Representation-floor and no-change baselines accompany endpoint results.